# Relatório de Extração de Dados: E-commerce (Web Scraping)
**Objetivo:** Este notebook documenta o processo de coleta automatizada de dados (web scraping) do catálogo de calçados do site Dafiti. O objetivo é construir a base de dados bruta que será posteriormente utilizada para modelagem preditiva e análise de preços. 

Para garantir a viabilidade da coleta em larga escala e evitar bloqueios por sistemas de segurança (Anti-Bot/WAF), o script implementa estratégias avançadas de simulação de navegação humana, como o uso de sessões persistentes, cabeçalhos customizados e pausas estocásticas.

In [ ]:
import os
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

## 1. Configuração de Navegação e Evasão de Bloqueios
Servidores modernos bloqueiam requisições que se comportam como robôs. Para mitigar isso, configuramos um dicionário de `headers` que mimetiza um navegador real (Google Chrome no Windows). 

Em seguida, inicializamos uma `requests.Session()`. Diferente de requisições isoladas, a sessão armazena *cookies* de navegação, fazendo com que o servidor interprete o acesso contínuo como um único usuário navegando pelo site, e não como um ataque de milhares de requisições independentes.

In [ ]:
# Headers para simular um navegador real e evitar bloqueios imediatos
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive",
    "Referer": "https://www.dafiti.com.br/"
}

# 1. Criação da Sessão (Fundamental para manter os cookies ativos!)
sessao = requests.Session()
sessao.headers.update(headers)

# Request inicial na página principal para capturar os cookies de "boas-vindas"
try:
    sessao.get("https://www.dafiti.com.br/", timeout=10)
    time.sleep(2)
except:
    pass

## 2. Definição do Escopo e Funções Auxiliares
Definimos a função modular `webscraping`, responsável por buscar tags HTML específicas de forma segura (retornando "N/A" caso o dado não exista na página, evitando quebras no código). 

Em seguida, mapeamos as categorias alvo (Feminino e Masculino) e limitamos a profundidade da varredura a 900 páginas por categoria.

In [ ]:
def webscraping(produto, parametro, chave):
    """
    Busca um elemento HTML específico dentro do bloco do produto.
    Retorna o texto limpo se encontrar, ou 'N/A' caso o elemento não exista.
    """
    alvo = produto.find(parametro, class_=chave)
    return alvo.get_text().strip() if alvo else "N/A"

# Mapeamento das URLs base. O '{}' servirá para injetar o número da página no loop.
categorias_alvo = [
    {"nome": "Feminino", "url": "https://www.dafiti.com.br/calcados-femininos/?page={}"},
    {"nome": "Masculino", "url": "https://www.dafiti.com.br/calcados-masculinos/?page={}"}
]

NUM_PAGINAS = 900 
todos_produtos = []

## 3. Loop de Extração Principal
Esta é a etapa central do *parser*. O algoritmo itera sobre cada categoria e suas respectivas páginas. O HTML retornado é processado pelo `BeautifulSoup`.

**Mecanismos de Defesa Adicionados:**
* **Debug Visual:** Se a página carregar (Status 200), mas não contiver os blocos de produto, o script assume que foi interceptado por um CAPTCHA. Ele salva o HTML em `pagina_erro_debug.html` para auditoria e interrompe o loop daquela categoria.
* **Pausas Dinâmicas:** Ao final de cada página, a função `random.uniform(2.5, 4.5)` insere um atraso aleatório, mascarando o padrão temporal característico de robôs.

In [ ]:
for categoria in categorias_alvo:
    print(f"\n--- Iniciando extração: {categoria['nome']} ---")
    
    for pagina in range(1, NUM_PAGINAS + 1):
        url_paginada = categoria['url'].format(pagina)
        
        # Realiza a requisição mantendo a sessão
        response = sessao.get(url_paginada)
        print(f"[{categoria['nome']}] Página {pagina}: Status {response.status_code}")
        
        if response.status_code != 200:
            print("Página não encontrada ou bloqueada. Indo para a próxima categoria.")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        bloco_infos = soup.find_all("div", class_="product-box-detail")
        
        # Sistema de Debug: Verifica se o site retornou uma página de bloqueio/CAPTCHA
        if not bloco_infos:
            print(f"Fim dos produtos na categoria {categoria['nome']} (ou bloqueio do site).")
            with open("pagina_erro_debug.html", "w", encoding="utf-8") as f:
                f.write(response.text)
            print("HTML salvo em 'pagina_erro_debug.html' para verificação de bloqueios.")
            break
            
        # Extração das informações de cada calçado na página
        for bloco in bloco_infos:
            entrada = {
                "Categoria": categoria['nome'],
                "Marca": webscraping(bloco, "div", "product-box-brand"),
                "Cor": webscraping(bloco, "div", "product-box-colors"),
                "Vendedor": webscraping(bloco, "div", "product-box-seller"),
                "Produto": webscraping(bloco, "p", "product-box-title"),
                "Tags": webscraping(bloco, "span", "product-box-category-name"),
                "Preço Original": webscraping(bloco, "span", "product-box-price-from"),
                "Preço Pix": webscraping(bloco, "span", "product-box-price-to"),
                "Preço Parcelado": webscraping(bloco, "span", "product-box-installment-value"),
                "Quantidade de Parcelas": webscraping(bloco, "span", "product-box-installment-parcel")
            }
            todos_produtos.append(entrada)
        
        # Pausa estocástica (2.5 a 4.5 segs) para evitar banimento de IP
        tempo_pausa = random.uniform(2.5, 4.5) 
        time.sleep(tempo_pausa) 

print(f"\nTotal coletado: {len(todos_produtos)} produtos.")

## 4. Consistência e Exportação de Dados
Após o término da coleta, a lista de dicionários é convertida nativamente em um `DataFrame` do Pandas. 

Para garantir a reprodutibilidade do projeto em outros ambientes (sem problemas de caminhos de arquivos quebrados), utilizamos a biblioteca `os` para mapear dinamicamente o diretório atual, retroceder um nível e criar estruturalmente a pasta `dados`. O dataset final é exportado no formato CSV.

In [ ]:
# Converte a lista em uma estrutura tabular
dados = pd.DataFrame(todos_produtos)

# 1. Mapeia o diretório de execução atual
pasta_atual = os.getcwd()

# 2. Constrói o caminho relativo (volta um nível e entra em 'dados')
caminho_pasta_dados = os.path.abspath(os.path.join(pasta_atual, '..', '00.Dados'))

# 3. Cria a pasta destino automaticamente, caso ela não exista
os.makedirs(caminho_pasta_dados, exist_ok=True)

# 4. Define o arquivo de saída
caminho_arquivo = os.path.join(caminho_pasta_dados, '202608_DadosDafiti_Combinados.csv')

# 5. Exporta o DataFrame
dados.to_csv(caminho_arquivo, index=False)

print(f"Exportação concluída com sucesso!")
print(f"Arquivo salvo em: {caminho_arquivo}")